# Create Academy of Medical Sciences (UK) Awards

Awards from the Academy of Medical Sciences (acmedsci.ac.uk), scraped from the
per-scheme awardee pages (server-rendered HTML). ~833 awardees across five
schemes: Springboard, Starter Grant for Clinical Lecturers, Newton
International Fellowship, Daniel Turnberg Travel Fellowship, and Networking
Grant. Each row has awardee name (split given/family), host institution, project
title, and the scheme (funder_scheme). Review-panel members are screened out in
the scraper.

The Academy does NOT publish award amounts for any scheme (UK fellowship
convention — Step 6.7 amount check waived), nor per-grant ids (funder_award_id
is slugify(scheme+name+institution)), nor start/end dates. funding_type is
fellowship for the Newton/Turnberg fellowships, grant otherwise. provenance
`acmedsci`, priority 323. F4320320241 (Path A). Distinct from the Chinese
Academy of Medical Sciences.

Source parquet: s3://openalex-ingest/awards/acmedsci/acmedsci_awards.parquet
Built by scripts/local/acmedsci_to_s3.py

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.acmedsci_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/acmedsci/acmedsci_awards.parquet`;


In [ ]:
%sql
-- Check row count (should be ~833)
SELECT COUNT(*) as total_awardees FROM openalex.awards.acmedsci_raw;

In [ ]:
%sql
-- Inspect column names
DESCRIBE openalex.awards.acmedsci_raw;

In [ ]:
%sql
-- Sample the raw data
SELECT * FROM openalex.awards.acmedsci_raw LIMIT 5;

## Step 2: Create AcMedSci Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.acmedsci_awards
USING delta
AS
WITH
the_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320241  -- Academy of Medical Sciences (UK)
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id,
        CAST(NULL AS DECIMAL(18,2)) as amount,           -- amounts not published (6.7 waiver)
        CAST(NULL AS STRING) as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        CASE WHEN g.funder_scheme LIKE '%Fellowship%' THEN 'fellowship' ELSE 'grant' END as funding_type,
        g.funder_scheme,
        'acmedsci' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        CAST(NULL AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        CAST(NULL AS STRING) as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.acmedsci_raw g
    CROSS JOIN the_funder f
)
SELECT * FROM awards_transformed;


## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'acmedsci' AND priority = 323;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    323 as priority
FROM openalex.awards.acmedsci_awards;

## Verification Queries

In [ ]:
%sql
-- Check row count
SELECT COUNT(*) as total_awards FROM openalex.awards.acmedsci_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type
FROM openalex.awards.acmedsci_awards LIMIT 10;

In [ ]:
%sql
-- Funder distribution (should all be Academy of Medical Sciences)
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.acmedsci_awards GROUP BY funder.display_name ORDER BY cnt DESC;

In [ ]:
%sql
-- Scheme distribution
SELECT funder_scheme, COUNT(*) as cnt
FROM openalex.awards.acmedsci_awards GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness
SELECT
  COUNT(*) as total,
  COUNT(display_name) as has_title,
  COUNT(lead_investigator) as has_pi,
  ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_with_title
FROM openalex.awards.acmedsci_awards;

In [ ]:
%sql
-- PI frequency check (Step 6.4a — top families should be a real long-tail)
SELECT lead_investigator.given_name as given, lead_investigator.family_name as family, COUNT(*) as n
FROM openalex.awards.acmedsci_awards
GROUP BY 1,2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Top institutions (should be real universities)
SELECT lead_investigator.affiliation.name as institution, COUNT(*) as cnt
FROM openalex.awards.acmedsci_awards
WHERE lead_investigator.affiliation.name IS NOT NULL
GROUP BY 1 ORDER BY cnt DESC LIMIT 20;

In [ ]:
%sql
-- Verify rows landed in openalex_awards_raw
SELECT COUNT(*) as acmedsci_in_raw
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'acmedsci' AND priority = 323;